# 05 Results Summary

Purpose: Aggregate saved Phase 2 results and regenerate final paper figures without rerunning inference.

Inputs: `results/phase2/evaluation_results.json`, `results/phase2/imbalance_results.json`, `results/phase2/continual_learning_results.json`.

Outputs: Regenerated figures in `figures/phase2/`, final summary table in notebook output, and `results/phase2/final_results_summary.csv`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

from src.phase2.config import get_phase2_config
from src.phase2.evaluation import load_results
from src.phase2.visualization import (
    plot_alpha_sensitivity,
    plot_confusion_matrices,
    plot_continual_learning_curve,
    plot_minority_f1_vs_imbalance,
    plot_phase2_vs_phase1,
    plot_scoring_comparison,
)

CONFIG = get_phase2_config()

REPO_ROOT = Path("../..").resolve()
EVAL_PATH = REPO_ROOT / "results" / "phase2" / "evaluation_results.json"
IMB_PATH = REPO_ROOT / "results" / "phase2" / "imbalance_results.json"
CONT_PATH = REPO_ROOT / "results" / "phase2" / "continual_learning_results.json"
FIG_DIR = REPO_ROOT / "figures" / "phase2"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
evaluation_results = load_results(str(EVAL_PATH))
imbalance_results = load_results(str(IMB_PATH))
continual_results = load_results(str(CONT_PATH))

In [ ]:
import csv

SUMMARY_CSV_PATH = REPO_ROOT / "results" / "phase2" / "final_results_summary.csv"
SUMMARY_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

ratio_key = "100"
local_ttr = (
    imbalance_results.get("variants", {})
    .get("local_dnds", {})
    .get(ratio_key, {})
    .get("per_class_f1", {})
    .get("TTR", 0.0)
)
other_ttr = [
    metrics.get(ratio_key, {}).get("per_class_f1", {}).get("TTR", 0.0)
    for name, metrics in imbalance_results.get("variants", {}).items()
    if name != "local_dnds"
]
best_other_ttr = max(other_ttr) if other_ttr else 0.0
gain = local_ttr - best_other_ttr

best_variant_name = max(
    evaluation_results["variants"],
    key=lambda name: evaluation_results["variants"][name].get("macro_f1", 0.0),
)

rows = []
for name, metrics in evaluation_results["variants"].items():
    rows.append(
        {
            "variant": name,
            "accuracy": metrics.get("accuracy", 0.0),
            "macro_f1": metrics.get("macro_f1", 0.0),
            "weighted_f1": metrics.get("weighted_f1", 0.0),
            "ttr_f1": metrics.get("per_class_f1", {}).get("TTR", 0.0),
            "inference_time_ms": metrics.get(
                "inference_time_ms", metrics.get("latency_ms_per_sample", 0.0)
            ),
            "best_alpha": evaluation_results.get("best_alpha", "n/a"),
            "ttr_gain_100_to_best_other": gain,
            "is_best_rac": int(name == best_variant_name),
        }
    )

rows.append(
    {
        "variant": "phase1_traditional_baseline",
        "accuracy": 0.8216,
        "macro_f1": 0.8177,
        "weighted_f1": 0.8177,
        "ttr_f1": 0.8177,
        "inference_time_ms": 1.43,
        "best_alpha": evaluation_results.get("best_alpha", "n/a"),
        "ttr_gain_100_to_best_other": gain,
        "is_best_rac": 0,
    }
)

headers = [
    "variant",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "ttr_f1",
    "inference_time_ms",
    "best_alpha",
    "ttr_gain_100_to_best_other",
    "is_best_rac",
]

with SUMMARY_CSV_PATH.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=headers)
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved consolidated final CSV: {SUMMARY_CSV_PATH}")

In [ ]:
plot_scoring_comparison(evaluation_results, str(FIG_DIR / "scoring_comparison.png"))
plot_alpha_sensitivity(
    evaluation_results["alpha_sweep"], str(FIG_DIR / "alpha_sensitivity.png")
)
plot_confusion_matrices(
    evaluation_results,
    CONFIG["class_names"],
    str(FIG_DIR / "confusion_matrices_phase2.png"),
)
plot_phase2_vs_phase1(
    max(
        v.get("macro_f1", 0.0)
        for k, v in evaluation_results["variants"].items()
        if k != "traditional"
    ),
    0.8177,
    str(FIG_DIR / "phase2_vs_phase1.png"),
)
plot_minority_f1_vs_imbalance(
    imbalance_results, str(FIG_DIR / "minority_f1_vs_imbalance.png")
)
plot_continual_learning_curve(
    continual_results, str(FIG_DIR / "continual_learning_curve.png")
)

In [ ]:
print("=" * 60)
print("PHASE 2 RAC EXPERIMENT - RESULTS SUMMARY")
print("=" * 60)
print("Variant              | Accuracy | Macro F1 | TTR F1 | Latency")
print("---------------------|----------|----------|--------|--------")
for name, metrics in evaluation_results["variants"].items():
    ttr_f1 = metrics.get("per_class_f1", {}).get("TTR", 0.0)
    latency = metrics.get(
        "inference_time_ms", metrics.get("latency_ms_per_sample", 0.0)
    )
    print(
        f"{name:<20} | {metrics['accuracy']*100:6.2f}% | {metrics['macro_f1']:.4f} | {ttr_f1:.4f} | {latency:.2f} ms"
    )

print(
    f"{'Phase 1 Traditional':<20} | {82.16:6.2f}% | {0.8177:.4f} | {0.8177:.4f} | {1.43:.2f} ms"
)
print("=" * 60)
print(f"Best alpha (local DNDS): {evaluation_results.get('best_alpha', 'n/a')}")

ratio_key = "100"
local_ttr = (
    imbalance_results.get("variants", {})
    .get("local_dnds", {})
    .get(ratio_key, {})
    .get("per_class_f1", {})
    .get("TTR", 0.0)
)
other_ttr = [
    metrics.get(ratio_key, {}).get("per_class_f1", {}).get("TTR", 0.0)
    for name, metrics in imbalance_results.get("variants", {}).items()
    if name != "local_dnds"
]
best_other_ttr = max(other_ttr) if other_ttr else 0.0
gain = local_ttr - best_other_ttr
print(f"Best imbalance correction gain on TTR F1 at 100:1 ratio: {gain:+.3f}")

phase1_per_class = {
    "Black": {"image": 0.5812, "text": 0.6491, "fused": 0.7141},
    "Blue": {"image": 0.7371, "text": 0.7501, "fused": 0.8220},
    "Green": {"image": 0.8334, "text": 0.8833, "fused": 0.9171},
    "TTR": {"image": 0.7084, "text": 0.8017, "fused": 0.8177},
}

best_variant_name = max(
    evaluation_results["variants"],
    key=lambda name: evaluation_results["variants"][name].get("macro_f1", 0.0),
)
best_variant_metrics = evaluation_results["variants"][best_variant_name]

print("")
print("Per-Class Performance (F1-Score)")
print("| Class  | Phase 1 Image | Phase 1 Text | Phase 1 Fused | Best RAC Variant |")
print("|--------|--------------|--------------|----------------|------------------|")
for cls in ["Black", "Blue", "Green", "TTR"]:
    rac_f1 = best_variant_metrics.get("per_class_f1", {}).get(cls, 0.0)
    print(
        f"| {cls:<6} | {phase1_per_class[cls]['image']:>12.4f} | "
        f"{phase1_per_class[cls]['text']:>12.4f} | {phase1_per_class[cls]['fused']:>14.4f} | {rac_f1:>16.4f} |"
    )

print(f"Best RAC variant in this run: {best_variant_name}")